# Exercise: Creative Writing Assistant
## AIAT 124 - Generative AI

## Learning Objectives

- Generate creative text
- Fine-tune language model
- Control generation style
- Apply to content creation

## Real-World Context

Build AI writing assistant for content creation, storytelling.

**Task**: Generate creative stories and articles.

---

## Task 1: Text Generation Setup (25 points)

Set up text generation:
1. Load pre-trained model
2. Prepare training data
3. Configure generation parameters

## 📥 Inputs & 📤 Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---

In [1]:
# ── Setup: pre-trained character LSTM language model ─────────────────────────
# (In real projects you'd load GPT-2 etc.; we use a local LSTM to avoid downloads.)
import torch, torch.nn as nn, torch.optim as optim, numpy as np
torch.manual_seed(0); np.random.seed(0)

corpus = ('the quick brown fox jumps over the lazy dog. ' * 20 +
          'hello world this is a test of the language model. ' * 20)
chars = sorted(set(corpus)); c2i={c:i for i,c in enumerate(chars)}
i2c={i:c for c,i in c2i.items()}; V,S=len(chars),30

X=torch.tensor([[c2i[corpus[j+k]] for k in range(S)] for j in range(len(corpus)-S)],dtype=torch.long)
y=torch.tensor([c2i[corpus[j+S]] for j in range(len(corpus)-S)],dtype=torch.long)

class CharLM(nn.Module):
    def __init__(self): super().__init__(); self.e=nn.Embedding(V,32); self.l=nn.LSTM(32,128,batch_first=True); self.f=nn.Linear(128,V)
    def forward(self,x): return self.f(self.l(self.e(x))[0][:,-1,:])

model=CharLM(); opt=optim.Adam(model.parameters(),lr=3e-3); crit=nn.CrossEntropyLoss()
from torch.utils.data import TensorDataset,DataLoader
loader=DataLoader(TensorDataset(X,y),batch_size=64,shuffle=True)
print('Training model...')
for ep in range(10):
    model.train(); el=0
    for xb,yb in loader:
        opt.zero_grad(); loss=crit(model(xb),yb); loss.backward(); opt.step(); el+=loss.item()
print(f'Done. Final loss={el/len(loader):.4f}')

# ── Task 1: Implement the generate() function ─────────────────────────────
# Fill in the body: given a `seed` string, generate `n` new characters.
# Hint: use temperature sampling: logits/temperature → softmax → np.random.choice
def generate(seed: str, n: int = 50, temperature: float = 1.0) -> str:
    model.eval()
    out = seed
    ctx = [c2i.get(c, 0) for c in seed[-S:]]
    for _ in range(n):
        x = torch.tensor([ctx], dtype=torch.long)
        with torch.no_grad():
            logits = model(x)[0] / max(temperature, 1e-6)
        probs = torch.softmax(logits, 0).numpy()
        nxt = int(np.random.choice(V, p=probs))
        out += i2c[nxt]
        ctx = ctx[1:] + [nxt]
    return out

# ── Task 2: Test your generate() with two temperature settings ─────────────
print('\nLow temperature (deterministic):')
print(generate('the quick', n=50, temperature=0.3))
print('\nHigh temperature (creative):')
print(generate('the quick', n=50, temperature=1.5))

Training model...


Done. Final loss=0.0106

Low temperature (deterministic):
the quick brown fox jumps over the lazy dog. the lazy dog. 

High temperature (creative):
the quick buela helc og. the tizs a of the lazy mof. hel o 


## Task 2: Fine-tuning (35 points)

Fine-tune model:
1. Prepare dataset
2. Configure training
3. Fine-tune on creative writing

# Fine-tuning stub — implement training loop for your creative-writing dataset.
# For grading, replace this with real fine-tune; for a quick sanity check you can overfit one batch.
print("Fine-tuning: TODO — add Trainer / loop here (see Course 10 Unit 2 examples).")

## Task 3: Generation and Evaluation (40 points)

Generate creative content and evaluate quality.

**Submission**: Complete notebook with writing assistant.

**Grading**: 100 points total